In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!apt-get update -qq && apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, time
env = os.environ.copy()
env["OLLAMA_NUM_PARALLEL"] = "4"
env["OLLAMA_MAX_LOADED_MODELS"] = "1"
env["OLLAMA_KEEP_ALIVE"] = "60m"
ollama_process = subprocess.Popen(["ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env)
time.sleep(8)
print("Ollama up with NUM_PARALLEL=4")
!ollama pull gemma2
print("Gemma2 ready")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out
W: Failed to fetch https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Some index files failed to download. They have been ignored, or old ones used instead.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user

In [4]:
# Numpy<2 requirement for spaCy. Restart kernel if needed.
import sys
need_restart = False
try:
    import numpy
    if int(numpy.__version__.split('.')[0]) >= 2:
        need_restart = True
except ImportError:
    pass

!pip install --quiet "numpy<2" "thinc<8.4" "spacy<3.8" scispacy
!pip install --quiet faiss-gpu-cu12 transformers tqdm
!pip install --quiet https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

if need_restart:
    print("\nNumpy was 2.x — restarting kernel. Re-run from cell 1.")
    os.kill(os.getpid(), 9)
else:
    print("Deps installed (numpy already <2)")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.2.5 requires numpy<2.0.0,>=1.19.0; python_version >= "3.9", but you have numpy 2.4.4 which is incompatible.
scispacy 0.6.2 requires numpy<2.0, but you have numpy 2.4.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
faiss-gpu-cu12 1.14.1.post1 requires numpy<3,>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=

In [5]:
import json, pickle, re, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import numpy as np
import requests
import torch
import torch.nn.functional as F
import faiss
import spacy
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}  ({torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'})")

DRIVE = Path("/content/drive/MyDrive/DL Project")
DATA = DRIVE / "synthetic_patients"
OLLAMA_URL = "http://localhost:11434/api/generate"
GEMMA_MODEL = "gemma2"
MAX_WORKERS = 4

Device: cuda  (NVIDIA A100-SXM4-40GB)


In [6]:
def load_jsonl(p):
    with open(p) as f:
        return [json.loads(l) for l in f if l.strip()]

patients = load_jsonl(DATA / "patients_v2.jsonl")
patient_by_id = {p["patient_id"]: p for p in patients}
train_q = load_jsonl(DATA / "questions_v2.jsonl")
test_q = load_jsonl(DATA / "test_questions_v2.jsonl")
all_questions = train_q + test_q
medline_articles = load_jsonl(DRIVE / "medlineplus_articles.jsonl")
print(f"patients: {len(patients)}  questions: {len(all_questions)}  "
      f"MedlinePlus articles: {len(medline_articles)}")

patients: 400  questions: 2000  MedlinePlus articles: 2156


In [7]:
models_dir = DRIVE / "models"
qtok = AutoTokenizer.from_pretrained(models_dir / "MedCPT-Query-Encoder")
qenc = AutoModel.from_pretrained(models_dir / "MedCPT-Query-Encoder").to(DEVICE).eval()
atok = AutoTokenizer.from_pretrained(models_dir / "MedCPT-Article-Encoder")
aenc = AutoModel.from_pretrained(models_dir / "MedCPT-Article-Encoder").to(DEVICE).eval()
ctok = AutoTokenizer.from_pretrained(models_dir / "MedCPT-Cross-Encoder")
cenc = AutoModelForSequenceClassification.from_pretrained(models_dir / "MedCPT-Cross-Encoder").to(DEVICE).eval()
print("MedCPT loaded")

nlp = spacy.load("en_ner_bc5cdr_md")
print("scispaCy loaded")

with open(DRIVE / "primekg_index.pkl", "rb") as f:
    name_to_triples = pickle.load(f)
print(f"PrimeKG: {len(name_to_triples):,} entities")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

MedCPT loaded


/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


scispaCy loaded
PrimeKG: 128,550 entities


In [8]:
def patient_to_chunks(p):
    pid, name, age, gender = p["patient_id"], p["name"], p["age"], p["gender"].lower()
    chunks = []
    conds = p.get("active_conditions", [])
    if conds:
        s = "; ".join(f"{c['condition']} (diagnosed {c['diagnosed']})" for c in conds)
        text = f"{name} is a {age}-year-old {gender} with the following active medical conditions: {s}."
    else:
        text = f"{name} is a {age}-year-old {gender} with no active medical conditions documented."
    chunks.append({"chunk_id": f"{pid}_dem_cond", "patient_id": pid, "section": "demographics_and_conditions", "text": text})
    pmh = p.get("past_medical_history", [])
    if pmh:
        chunks.append({"chunk_id": f"{pid}_pmh", "patient_id": pid, "section": "past_medical_history", "text": f"{name} has the following past medical and surgical history: " + "; ".join(pmh) + "."})
    al = p.get("allergies", [])
    if al:
        text = f"{name} has the following documented drug or substance allergies: " + "; ".join(f"{a['substance']} ({a.get('reaction','unspecified')})" for a in al) + "."
    else:
        text = f"{name} has no documented drug allergies."
    chunks.append({"chunk_id": f"{pid}_allergies", "patient_id": pid, "section": "allergies", "text": text})
    v = p.get("recent_vitals", {})
    if v:
        date = v.get("date", "recent visit")
        parts = [f"{k.replace('_',' ')} {val}" for k, val in v.items() if k != "date"]
        chunks.append({"chunk_id": f"{pid}_vitals", "patient_id": pid, "section": "recent_vitals", "text": f"{name}'s recent vitals from {date}: " + ", ".join(parts) + "."})
    ls = p.get("lifestyle", {})
    if ls:
        parts = [f"{k.replace('_',' ')}: {val}" for k, val in ls.items()]
        chunks.append({"chunk_id": f"{pid}_lifestyle", "patient_id": pid, "section": "lifestyle", "text": f"{name}'s lifestyle factors. " + ". ".join(parts) + "."})
    return chunks

all_patient_chunks = []
for p in tqdm(patients, desc="chunking patients"):
    all_patient_chunks.extend(patient_to_chunks(p))
print(f"Patient chunks: {len(all_patient_chunks)}")

def encode_batch(texts, tok, enc, batch_size=32, max_length=256):
    embs = []
    for i in range(0, len(texts), batch_size):
        b = texts[i:i+batch_size]
        e = tok(b, truncation=True, padding=True, max_length=max_length, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            o = enc(**e)
            v = F.normalize(o.last_hidden_state[:,0,:], dim=-1)
        embs.append(v.cpu().numpy())
    return np.vstack(embs)

p_embs = encode_batch([c["text"] for c in all_patient_chunks], atok, aenc)
patient_chunks_by_id, patient_embs_by_id = {}, {}
for c, e in zip(all_patient_chunks, p_embs):
    patient_chunks_by_id.setdefault(c["patient_id"], []).append(c)
    patient_embs_by_id.setdefault(c["patient_id"], []).append(e)
for pid in patient_embs_by_id:
    patient_embs_by_id[pid] = np.array(patient_embs_by_id[pid])
print(f"Per-patient embeddings ready for {len(patient_embs_by_id)} patients")

chunking patients:   0%|          | 0/400 [00:00<?, ?it/s]

Patient chunks: 1633
Per-patient embeddings ready for 400 patients


In [9]:
def chunk_article(article, target_words=220):
    paragraphs = [p.strip() for p in article["text"].split("\n") if p.strip()]
    chunks = []
    cur_words, cur_paras = [], []
    for p in paragraphs:
        pw = p.split()
        if len(cur_words) + len(pw) > target_words and cur_words:
            chunks.append("\n".join(cur_paras))
            cur_words, cur_paras = [], []
        cur_words.extend(pw)
        cur_paras.append(p)
    if cur_words:
        chunks.append("\n".join(cur_paras))
    return [{"chunk_id": f"{article['article_id']}_c{i}", "article_id": article["article_id"],
             "title": article["title"], "url": article["url"],
             "text": txt, "n_words": len(txt.split())} for i, txt in enumerate(chunks)]

all_med_chunks = []
for art in tqdm(medline_articles, desc="chunking medlineplus"):
    all_med_chunks.extend(chunk_article(art))
print(f"MedlinePlus chunks: {len(all_med_chunks)}")

med_embs = encode_batch([c["text"] for c in all_med_chunks], atok, aenc)
print(f"MedlinePlus embeddings: {med_embs.shape}")

chunking medlineplus:   0%|          | 0/2156 [00:00<?, ?it/s]

MedlinePlus chunks: 15803
MedlinePlus embeddings: (15803, 768)


In [10]:
PRED = {"indication":"is indicated for","contraindication":"is contraindicated in","side effect":"can cause","drug-drug interaction":"interacts with","synergistic interaction":"synergistically interacts with","target":"targets","enzyme":"is metabolized by","phenotype present":"presents with","associated with":"is associated with"}

def encode_query(q):
    e = qtok([q], truncation=True, padding=True, max_length=64, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        o = qenc(**e)
        v = F.normalize(o.last_hidden_state[:,0,:], dim=-1)
    return v.cpu().numpy()

def retrieve_patient(q, pid, k=4):
    if pid not in patient_embs_by_id: return []
    qv = encode_query(q)
    s = (patient_embs_by_id[pid] @ qv.T).flatten()
    chunks = patient_chunks_by_id[pid]
    idx = np.argsort(-s)[:k]
    return [{"score": float(s[i]), "chunk_id": chunks[i]["chunk_id"], "section": chunks[i]["section"], "text": chunks[i]["text"]} for i in idx]

def retrieve_medlineplus(q, k=10):
    qv = encode_query(q)
    s = (med_embs @ qv.T).flatten()
    idx = np.argsort(-s)[:k]
    return [{"score": float(s[i]), "chunk_id": all_med_chunks[i]["chunk_id"],
             "title": all_med_chunks[i]["title"], "url": all_med_chunks[i]["url"],
             "text": all_med_chunks[i]["text"]} for i in idx]

def rerank(q, cands, k=3):
    if not cands: return []
    pairs = [[q, c["text"]] for c in cands]
    scores = []
    for i in range(0, len(pairs), 8):
        b = pairs[i:i+8]
        e = ctok(b, truncation=True, padding=True, max_length=512, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            l = cenc(**e).logits.squeeze(dim=-1)
        if l.dim()==0: scores.append(float(l))
        else: scores.extend(l.cpu().tolist())
    for c, s in zip(cands, scores): c["cross_encoder_score"] = float(s)
    return sorted(cands, key=lambda x: x["cross_encoder_score"], reverse=True)[:k]

def kg_for_drugs(drugs, k=4):
    triples, seen = [], set()
    for d in drugs:
        for v in [d.lower(), d.lower().split()[0]]:
            if v in name_to_triples:
                for rel, y, t in name_to_triples[v][:k]:
                    key = (v, rel, y)
                    if key in seen: continue
                    seen.add(key)
                    triples.append({"subject": v, "predicate": rel, "object": y, "object_type": t, "triple_text": f"{v} {rel} {y}"})
                break
    return triples

def kg_for_question(q, k=3):
    triples, seen = [], set()
    for ent in nlp(q).ents:
        e = ent.text.strip().lower()
        for v in (e, e.split()[0] if e.split() else "", e.replace("-"," ").split()[0] if e else ""):
            if not v or v in seen: continue
            if v in name_to_triples:
                for rel, y, t in name_to_triples[v][:k]:
                    key = (v, rel, y)
                    if key in seen: continue
                    seen.add(key)
                    triples.append({"subject": v, "predicate": rel, "object": y, "object_type": t, "triple_text": f"{v} {rel} {y}"})
    for w in re.findall(r"\b[a-z]{5,}\b", q.lower()):
        if len(triples) >= 12: break
        if w in name_to_triples:
            for rel, y, t in name_to_triples[w][:2]:
                key = (w, rel, y)
                if key in seen: continue
                seen.add(key)
                triples.append({"subject": w, "predicate": rel, "object": y, "object_type": t, "triple_text": f"{w} {rel} {y}"})
    return triples

def fmt_rx(p):
    m = p.get("medications", [])
    if not m: return "(no active prescriptions on file)"
    return "\n".join(f"- {x['drug']} {x['dosage']}, {x['frequency']}, for {x['indication']}" + (f" (Note: {x['notes']})" if x.get('notes') else "") for x in m)

def fmt_pctx(rk):
    return "=== PATIENT MEDICAL RECORD ===\n" + "\n".join(f"- {c['text']}" for c in rk) if rk else ""

def fmt_kg(tr):
    if not tr: return ""
    out = []
    for t in tr:
        ph = PRED.get(t['predicate'].lower(), t['predicate'])
        out.append(f"- {t['subject'].capitalize()} {ph} {t['object']}.")
    return "=== DRUG SAFETY INFORMATION (from validated drug databases) ===\n" + "\n".join(out)

def build_sources_for_mode2(patient_chunks, med_chunks, patient):
    sources, sid = [], 1
    if patient.get("medications"):
        sources.append({"id": sid, "label": "Patient prescriptions", "text": fmt_rx(patient), "kind": "prescriptions", "url": ""})
        sid += 1
    for c in patient_chunks:
        sources.append({"id": sid, "label": f"Patient record ({c['section'].replace('_', ' ')})", "text": c["text"], "kind": "patient", "url": ""})
        sid += 1
    for c in med_chunks:
        sources.append({"id": sid, "label": f"MedlinePlus: {c['title']}", "text": c["text"], "kind": "medlineplus", "url": c.get("url", "")})
        sid += 1
    return sources

def fmt_cited_ctx(sources):
    if not sources: return ""
    parts = ["=== REFERENCE MATERIAL (cite inline as [N]) ==="]
    for s in sources:
        parts.append(f"[{s['id']}] {s['label']}\n{s['text']}")
    return "\n\n".join(parts)

def append_sources_block(answer, sources):
    if not sources: return answer
    cited = sorted({s["id"] for s in sources if f"[{s['id']}]" in answer})
    listed = cited if cited else [s["id"] for s in sources]
    lines = ["", "**Sources**"]
    for sid in listed:
        s = next((x for x in sources if x["id"] == sid), None)
        if s is None: continue
        if s["url"]: lines.append(f"[{sid}] {s['label']} — <{s['url']}>")
        else: lines.append(f"[{sid}] {s['label']}")
    return answer + "\n" + "\n".join(lines)

def prompt_v1(q, p, ctx="", kg=""):
    parts = ["You are a helpful medical assistant answering questions for a patient about their",
             "prescriptions and health. Use the information provided to give a clear, accurate,",
             "and patient-friendly answer. If you do not have the information needed, say so",
             "honestly rather than guessing.", "", "=== ACTIVE PRESCRIPTIONS ===", fmt_rx(p), ""]
    if ctx: parts += [ctx, ""]
    if kg: parts += [kg, ""]
    parts += ["=== PATIENT QUESTION ===", q, "", "Provide a concise, helpful answer:"]
    return "\n".join(parts)

def prompt_mode2_with_citations(q, ctx):
    return ("You are a helpful medical assistant answering questions for a patient about\n"
            "their prescriptions and health. The numbered references below are the ONLY\n"
            "things you may cite. Read each reference carefully.\n\n"
            "Citation rules (strict):\n"
            "1. After each factual claim, add a citation like [1] or [3].\n"
            "2. Cite a reference ONLY if its text directly states or supports the claim.\n"
            "3. Do not invent reference numbers. Do not cite [N] for a claim if [N] is unrelated.\n"
            "4. You may cite multiple references for one claim ([1][3]).\n"
            "5. If none of the references answer the question, say so honestly.\n\n"
            f"{ctx}\n\n=== PATIENT QUESTION ===\n{q}\n\n"
            "Provide a concise, helpful answer using only the references above for citations:")

def prompt_v2_mode3(q, p, ctx, kg):
    return ("You are a helpful medical assistant answering questions for a patient about their\n"
            "prescriptions and health. Use the patient's prescriptions, medical record, and the\n"
            "drug safety information below to give a thoughtful, helpful answer.\n\n"
            "If the safety information directly addresses the question, share it in plain language.\n"
            "Recommending the patient confirm with their doctor is appropriate, but try to be\n"
            "informative first rather than only deferring. When the medical facts above clearly\n"
            "answer the question, lead with that information.\n\n"
            f"=== ACTIVE PRESCRIPTIONS ===\n{fmt_rx(p)}\n\n{ctx}\n\n{kg}\n\n"
            f"=== PATIENT QUESTION ===\n{q}\n\nProvide a concise, helpful answer:")

def query_gemma(prompt, temperature=0.0, max_tokens=512):
    try:
        r = requests.post(OLLAMA_URL, json={"model": GEMMA_MODEL, "prompt": prompt, "stream": False,
                                              "options": {"temperature": temperature, "num_predict": max_tokens, "seed": 42}}, timeout=300)
        r.raise_for_status()
        return r.json().get("response", "").strip()
    except Exception as e:
        return f"[ollama error: {e}]"

def mode_1(q, p):
    t = time.time()
    a = query_gemma(prompt_v1(q, p))
    return {"mode":1, "mode_name":"LLM_only", "answer":a, "latency_seconds":round(time.time()-t,2),
            "n_retrieved_chunks":0, "n_medlineplus_chunks":0, "n_kg_triples":0,
            "retrieved_chunks":[], "medlineplus_chunks":[], "kg_triples":[]}

def mode_2(q, p):
    t = time.time()
    cand_p = retrieve_patient(q, p["patient_id"], k=4)
    rk_p = rerank(q, cand_p, k=3)
    cand_m = retrieve_medlineplus(q, k=10)
    rk_m_all = rerank(q, cand_m, k=10) if cand_m else []
    rk_m_filtered = [c for c in rk_m_all if c.get("cross_encoder_score", c["score"]) > -3.0][:2]
    sources = build_sources_for_mode2(rk_p, rk_m_filtered, p)
    ctx = fmt_cited_ctx(sources)
    answer = query_gemma(prompt_mode2_with_citations(q, ctx))
    answer_with_refs = append_sources_block(answer, sources)
    return {"mode":2, "mode_name":"RAG", "answer":answer_with_refs, "latency_seconds":round(time.time()-t,2),
            "n_retrieved_chunks":len(rk_p), "n_medlineplus_chunks":len(rk_m_filtered), "n_kg_triples":0,
            "retrieved_chunks":[{"section":c["section"], "score":c.get("cross_encoder_score",c["score"]), "text":c["text"][:200]} for c in rk_p],
            "medlineplus_chunks":[{"title":c["title"], "url":c["url"], "score":c.get("cross_encoder_score",c["score"]), "text":c["text"][:200]} for c in rk_m_filtered],
            "kg_triples":[]}

def mode_3(q, p):
    t = time.time()
    cands = retrieve_patient(q, p["patient_id"], k=4)
    rk = rerank(q, cands, k=3)
    drugs = [m["drug"] for m in p.get("medications", [])]
    triples, seen = [], set()
    for tr in kg_for_drugs(drugs, k=4) + kg_for_question(q, k=3):
        key = (tr["subject"], tr["predicate"], tr["object"])
        if key in seen: continue
        seen.add(key)
        triples.append(tr)
    a = query_gemma(prompt_v2_mode3(q, p, fmt_pctx(rk), fmt_kg(triples)))
    return {"mode":3, "mode_name":"RAG_KG", "answer":a, "latency_seconds":round(time.time()-t,2),
            "n_retrieved_chunks":len(rk), "n_medlineplus_chunks":0, "n_kg_triples":len(triples),
            "retrieved_chunks":[{"section":c["section"], "score":c.get("cross_encoder_score",c["score"]), "text":c["text"][:200]} for c in rk],
            "medlineplus_chunks":[],
            "kg_triples":[t["triple_text"] for t in triples]}

print("Pipelines ready (Mode 2 = patient RAG + MedlinePlus + citations)")

Pipelines ready (Mode 2 = patient RAG + MedlinePlus + citations)


In [11]:
import os, subprocess, time
env = os.environ.copy()
env["OLLAMA_NUM_PARALLEL"] = "4"
env["OLLAMA_MAX_LOADED_MODELS"] = "1"
env["OLLAMA_KEEP_ALIVE"] = "60m"
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env=env,
)
time.sleep(8)
print("Ollama restarted with NUM_PARALLEL=4")
# Verify it's responding
import requests
r = requests.get("http://localhost:11434/api/tags", timeout=5)
print(f"Ollama tags response: {r.status_code}")

Ollama restarted with NUM_PARALLEL=4
Ollama tags response: 200


In [12]:
for q in all_questions[:3]:
    p = patient_by_id[q["patient_id"]]
    r1 = mode_1(q["question"], p)
    r2 = mode_2(q["question"], p)
    r3 = mode_3(q["question"], p)
    print(f"\n{q['question_id']} ({q['category']}, expected M{q['expected_mode']})")
    print(f"  M1 ({r1['latency_seconds']}s): {r1['answer'][:120]}…")
    print(f"  M2 ({r2['latency_seconds']}s) [{r2['n_retrieved_chunks']}p+{r2['n_medlineplus_chunks']}m chunks]:")
    print(f"    {r2['answer'][:200]}…")
    print(f"  M3 ({r3['latency_seconds']}s) [{r3['n_retrieved_chunks']}p+{r3['n_kg_triples']}kg]:")
    print(f"    {r3['answer'][:200]}…")
print("\nIf the M2 answers contain [1]/[2]/etc. inline citations and a Sources block at the end, proceed to cell 7.")


Q001 (drug_mechanism, expected M1)
  M1 (83.71s): Metformin is a medication used to help manage Type 2 diabetes. It works by helping your body use insulin more effectivel…
  M2 (1.37s) [3p+2m chunks]:
    Metformin is used to treat type 2 diabetes [5]. It helps control the amount of glucose in your blood by decreasing the amount absorbed from food and made by the liver [5]. Metformin also increases you…
  M3 (1.31s) [3p+8kg]:
    Metformin is a medication used to manage Type 2 diabetes. It works by targeting several pathways in your body that affect blood sugar levels.  It helps your body use insulin more effectively and reduc…

Q002 (patient_timing, expected M2)
  M1 (0.63s): You should take your Metformin twice a day, with meals.  This helps reduce any stomach upset you might experience.…
  M2 (0.84s) [3p+2m chunks]:
    You should take your Metformin twice daily with meals [1].

**Sources**
[1] Patient prescriptions…
  M3 (0.76s) [3p+8kg]:
    Your Metformin prescription says to ta

In [12]:
responses_path = DATA / "patient_portal_responses_v3_2000.jsonl"

done_ids = set()
if responses_path.exists():
    with open(responses_path) as f:
        for line in f:
            try: done_ids.add(json.loads(line)["question_id"])
            except Exception: pass
    print(f"Resuming: {len(done_ids)} done")

remaining = [q for q in all_questions if q["question_id"] not in done_ids]
print(f"To process: {len(remaining)}")

def run_one(q):
    p = patient_by_id[q["patient_id"]]
    try:
        r1 = mode_1(q["question"], p)
        r2 = mode_2(q["question"], p)
        r3 = mode_3(q["question"], p)
        return {"question_id": q["question_id"], "patient_id": q["patient_id"],
                "question": q["question"], "category": q["category"],
                "expected_mode": q["expected_mode"],
                "expected_mode_name": q.get("expected_mode_name", ""),
                "rationale": q.get("rationale", ""),
                "mode_1": r1, "mode_2": r2, "mode_3": r3}
    except Exception as e:
        return {"question_id": q["question_id"], "error": str(e)}

CHECKPOINT_EVERY = 50
completed, errors = 0, 0
t0 = time.time()
with open(responses_path, "a") as fout:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(run_one, q): q for q in remaining}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"v3 (M2=RAG+MedPlus+cite, {MAX_WORKERS}-way)"):
            rec = fut.result()
            if rec.get("error"):
                errors += 1; continue
            fout.write(json.dumps(rec) + "\n")
            completed += 1
            if completed % CHECKPOINT_EVERY == 0:
                fout.flush()
                el = time.time() - t0
                rate = completed / el
                eta = (len(futures) - completed) / rate if rate > 0 else 0
                print(f"\n[ckpt] {completed}/{len(futures)}  {rate:.2f} q/s  ETA {eta/60:.1f}min  errors {errors}")

print(f"\nDone. {completed} written, {errors} errors. {(time.time()-t0)/60:.1f} min total.")
print(f"Saved to {responses_path}")

Resuming: 1350 done
To process: 650


v3 (M2=RAG+MedPlus+cite, 4-way):   0%|          | 0/650 [00:00<?, ?it/s]


[ckpt] 50/650  0.15 q/s  ETA 64.9min  errors 0

[ckpt] 100/650  0.18 q/s  ETA 49.7min  errors 0

[ckpt] 150/650  0.20 q/s  ETA 41.2min  errors 0

[ckpt] 200/650  0.20 q/s  ETA 37.0min  errors 0

[ckpt] 250/650  0.20 q/s  ETA 34.1min  errors 0

[ckpt] 300/650  0.19 q/s  ETA 30.0min  errors 0

[ckpt] 350/650  0.19 q/s  ETA 25.8min  errors 0

[ckpt] 400/650  0.20 q/s  ETA 21.2min  errors 0

[ckpt] 450/650  0.20 q/s  ETA 17.0min  errors 0

[ckpt] 500/650  0.20 q/s  ETA 12.6min  errors 0

[ckpt] 550/650  0.20 q/s  ETA 8.4min  errors 0

[ckpt] 600/650  0.20 q/s  ETA 4.2min  errors 0

[ckpt] 650/650  0.20 q/s  ETA 0.0min  errors 0

Done. 650 written, 0 errors. 53.8 min total.
Saved to /content/drive/MyDrive/DL Project/synthetic_patients/patient_portal_responses_v3_2000.jsonl


In [14]:
import json
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/DL Project")
DATA = DRIVE / "synthetic_patients"

for fname in ["patient_portal_responses_v3_2000.jsonl",
              "patient_portal_ground_truth_v3_2000.jsonl"]:
    path = DATA / fname
    seen = {}
    with open(path) as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                seen[r["question_id"]] = r  # last-write-wins
    with open(path, "w") as f:
        for r in seen.values():
            f.write(json.dumps(r) + "\n")
    print(f"{fname}: deduped to {len(seen)} unique records")

patient_portal_responses_v3_2000.jsonl: deduped to 2000 unique records
patient_portal_ground_truth_v3_2000.jsonl: deduped to 2000 unique records


In [15]:
PUNT_RE = re.compile(r"i can'?t give (you )?medical advice|i'?m sorry,? but i can'?t|i am not (a doctor|a medical|able to)|please (talk to|consult|speak to|discuss with) (your )?doctor|only your doctor can", re.IGNORECASE)

def is_pure_punt(answer):
    if not answer: return True
    if len(answer) < 250 and PUNT_RE.search(answer):
        useful = sum(1 for s in re.split(r"[.!?]+", answer) if len(s.strip()) >= 15 and not PUNT_RE.search(s))
        return useful <= 1
    return False

def cites_specific_value(a, p):
    al = a.lower()
    for k, v in p.get("recent_vitals", {}).items():
        if k == "date": continue
        if str(v).lower() in al: return True
    for x in p.get("allergies", []):
        if x.get("substance", "").lower() in al: return True
    for h in p.get("past_medical_history", []):
        if h.lower().split()[:3] and " ".join(h.lower().split()[:3]) in al: return True
    return False

def mentions_interaction(a):
    al = a.lower()
    return any(k in al for k in ["interact", "increase the levels", "decrease the levels", "bleeding risk", "cyp", "metabolism", "hyperkalemi", "potassium", "potentiate", "could interact", "may interact"])

def gives_safety_advice(a):
    al = a.lower()
    return any(re.search(p, al) for p in [r"avoid", r"don'?t (take|drink|combine)", r"safe to take", r"can take .* with", r"should not", r"contraindicated", r"caution", r"may cause", r"could interact"])

def is_informative_diplomatic(a, p):
    if len(a) < 200: return False
    al = a.lower()
    has_specific = mentions_interaction(a)
    if not has_specific:
        for m in p.get("medications", []):
            if m["drug"].split()[0].lower() in al and len(m["drug"].split()[0]) > 3:
                has_specific = True; break
    return has_specific and sum(1 for s in re.split(r"[.!?]+", a) if len(s.strip()) >= 25 and not PUNT_RE.search(s.lower())) >= 2

def score_mode(rec, p, expected, rationale):
    a = rec["answer"]
    if not a or len(a) < 30: return 0, "empty/short"
    if is_pure_punt(a): return 0, "punted"
    m = rec["mode"]
    if expected == "LLM_only": return 1, "generic OK for any mode"
    if expected == "RAG":
        if m == 1:
            r = rationale.lower()
            if any(k in r for k in ["notes", "prescription", "indication"]) and any(x["drug"].lower() in a.lower() for x in p.get("medications", [])):
                return 1, "answer in prescription text"
            if any(k in r for k in ["vital", "actual value", "history", "patient context"]):
                return (1, "got specific value") if cites_specific_value(a, p) else (0, "needed patient data")
            if "lifestyle" in r: return 0, "needed lifestyle context"
            return (1, "reasonable") if len(a) > 100 else (0, "too generic")
        return 1, "patient context applied"
    if expected == "RAG_KG":
        if mentions_interaction(a) and gives_safety_advice(a): return 1, "interaction + concrete advice"
        if gives_safety_advice(a): return 1, "concrete safety advice"
        if is_informative_diplomatic(a, p): return 1, "diplomatic but informative"
        return 0, "missed safety/interaction"
    return 0, "unknown"

def score_record(r, p):
    exp = r["expected_mode_name"]
    rt = r["rationale"]
    s1, w1 = score_mode(r["mode_1"], p, exp, rt)
    s2, w2 = score_mode(r["mode_2"], p, exp, rt)
    s3, w3 = score_mode(r["mode_3"], p, exp, rt)
    winner = 1 if s1 else (2 if s2 else (3 if s3 else 0))
    return {"question_id": r["question_id"], "patient_id": r["patient_id"], "category": r["category"],
            "expected_mode": r["expected_mode"], "expected_mode_name": r["expected_mode_name"],
            "mode_1_correct": s1, "mode_1_rationale": w1,
            "mode_2_correct": s2, "mode_2_rationale": w2,
            "mode_3_correct": s3, "mode_3_rationale": w3,
            "winning_mode": winner}

responses_all = load_jsonl(responses_path)
print(f"Loaded {len(responses_all)} responses")

gt = []
for r in tqdm(responses_all, desc="scoring"):
    p = patient_by_id[r["patient_id"]]
    gt.append(score_record(r, p))

gt_path = DATA / "patient_portal_ground_truth_v3_2000.jsonl"
with open(gt_path, "w") as f:
    for g in gt: f.write(json.dumps(g) + "\n")
print(f"\nSaved {gt_path}")

from collections import Counter
n = len(gt)
print(f"\nMode 1: {sum(g['mode_1_correct'] for g in gt)/n*100:.1f}%")
print(f"Mode 2 (with MedlinePlus): {sum(g['mode_2_correct'] for g in gt)/n*100:.1f}%")
print(f"Mode 3: {sum(g['mode_3_correct'] for g in gt)/n*100:.1f}%")
print(f"Oracle: {sum(1 for g in gt if g['mode_1_correct'] or g['mode_2_correct'] or g['mode_3_correct'])/n*100:.1f}%")
print("\nWinning mode distribution:", Counter(g['winning_mode'] for g in gt).most_common())
print("\nDownload these to your Mac then re-run scripts/retrain_router_v3.py:")
print(f"  {responses_path}")
print(f"  {gt_path}")

Loaded 2000 responses


scoring:   0%|          | 0/2000 [00:00<?, ?it/s]


Saved /content/drive/MyDrive/DL Project/synthetic_patients/patient_portal_ground_truth_v3_2000.jsonl

Mode 1: 84.2%
Mode 2 (with MedlinePlus): 92.5%
Mode 3: 98.3%
Oracle: 99.6%

Winning mode distribution: [(1, 1684), (2, 263), (3, 45), (0, 8)]

Download these to your Mac then re-run scripts/retrain_router_v3.py:
  /content/drive/MyDrive/DL Project/synthetic_patients/patient_portal_responses_v3_2000.jsonl
  /content/drive/MyDrive/DL Project/synthetic_patients/patient_portal_ground_truth_v3_2000.jsonl
